# Square: train, evaluate and visualize one policy

*Author: Mohammad Al-Jarrah*

Run with the Square environment (`environment_square.yml`, see the README). Choose a policy and the two modifications of Table 3:

- `STATE_NOISE`: train with Gaussian noise (std 0.02, normalized units) on the object pose in the conditioning states;
- `MEAN_OF`: execute the mean of 8 sampled chunks instead of one sample.

The simulator runs on the CPU and an episode takes several seconds; for the full protocol of the paper use `scripts/evaluate.py --workers N`.

In [ ]:
import os, sys, json
os.environ.setdefault('SDL_VIDEODRIVER', 'dummy')            # headless rendering of Push-T
ROOT = os.path.abspath('..')                                  # the repository root
sys.path.insert(0, ROOT)

import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import Image as ShowImage, display

import experiment as X
import visualization as V
from policies import NAMES
fontsize = 11
labeling = True                                               # set False to hide axis labels

## Settings

In [ ]:
POLICY      = 'otp'     # otp | fmp | sip | krp | dp | dpr | cp
CP_STEPS    = 1         # CP only: 1 or 3
STATE_NOISE = False     # checkpoint trained with state noise
MEAN_OF     = 1         # 1, or 8 for the mean of 8 samples
MODE        = 'train'   # 'train' or 'load'
SEED        = 0
N_EPISODES  = 20
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cfg = X.get_config('square', POLICY, STATE_NOISE)
print(NAMES[POLICY], 'settings:', cfg)

## Train (or load)

In [ ]:
ckpt = X.checkpoint_path('square', POLICY, SEED, STATE_NOISE)
if MODE == 'train':
    ckpt, seconds = X.train('square', POLICY, seed=SEED, state_noise=STATE_NOISE, device=DEVICE)
assert os.path.exists(ckpt), 'no checkpoint at %s: set MODE = "train"' % ckpt
print('checkpoint:', ckpt)

## Evaluate
The score of an episode is 1 if the nut is placed on the peg within 400 steps, else 0.

In [ ]:
agent, cfg = X.make_agent('square', POLICY, ckpt, DEVICE, mean_of=MEAN_OF, cp_steps=CP_STEPS)
episodes = X.task_module('square').HELD_OUT_SEEDS[:N_EPISODES]
T = X.task_module('square')
env = T.make_env()
actions = T.load_demos()['action']
bounds = (actions.min(0), actions.max(0))
results = []
for k, e in enumerate(episodes):
    agent.reset(e)
    results.append(T.rollout(env, agent, seed=e, h_o=cfg['h_o'], h_a=cfg['h_a'], bounds=bounds))
    print('episode %d/%d  %s in %d steps' % (k + 1, len(episodes), 'success' if results[-1]['success'] else 'failure',
                                             results[-1]['steps']))
summary = X.summarize([r['score'] for r in results])
print('%s: success rate %.3f +- %.3f over %d episodes' % (NAMES[POLICY], summary['mean'], summary['se'], summary['n']))

## Visualize
Top-down view of the table: the square peg (green), the nut at the start (gray, with its handle) and at the end of the episode (dashed), and the end effector at the start (circle).

In [ ]:
fig = V.square_scenes([r['start_obs'] for r in results], [r['end_obs'] for r in results],
                      [r['success'] for r in results], cols=5)
plt.show()